In [1]:
import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
import os
from sklearn.metrics import f1_score

# ---------------------- 1. 环境配置 ----------------------
def set_chinese_font():
    try:
        font_paths = ['/usr/share/fonts/truetype/wqy/wqy-microhei.ttc',
                      'C:/Windows/Fonts/simhei.ttf',
                      '/System/Library/Fonts/PingFang.ttc']
        for font_path in font_paths:
            if os.path.exists(font_path):
                fm.fontManager.addfont(font_path)
                plt.rcParams['font.family'] = fm.FontProperties(fname=font_path).get_name()
                plt.rcParams['axes.unicode_minus'] = False
                return True
        return False
    except: return False
set_chinese_font()

# ---------------------- 2. 标准 Transformer 核心组件 ----------------------

class Transformer_EncoderLayer_Standard(nn.Module):
    """使用标准多头自注意力的编码器层 (替代 FEDformer 的 FEB)"""
    def __init__(self, d_model, nhead, dropout=0.1):
        super(Transformer_EncoderLayer_Standard, self).__init__()
        # 标准时域自注意力机制
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(d_model * 4, d_model)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # 采用与 FEDformer 一致的 Pre-Norm 结构
        res = x
        x = self.norm1(x)
        attn_output, _ = self.self_attn(x, x, x)
        x = res + self.dropout(attn_output)
        
        res = x
        x = self.norm2(x)
        x = res + self.dropout(self.ffn(x))
        return x

# ---------------------- 3. 标准 Transformer 多任务预测模型 ----------------------

class Transformer_Satellite_Baseline(nn.Module):
    def __init__(self, input_dim=10, d_model=512, nhead=16, num_layers=4, horizon=10, seq_len=12):
        super(Transformer_Satellite_Baseline, self).__init__()
        self.horizon = horizon
        
        # 保持与 FEDformer 一致的多尺度特征提取 (MSFF)
        self.conv3 = nn.Conv1d(input_dim, d_model // 4, kernel_size=3, padding=1)
        self.conv5 = nn.Conv1d(input_dim, d_model // 4, kernel_size=5, padding=2)
        self.conv7 = nn.Conv1d(input_dim, d_model // 2, kernel_size=7, padding=3)
        self.bn = nn.BatchNorm1d(d_model)
        self.gelu = nn.GELU()
        
        # 标准位置编码
        self.pos_emb = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        
        # 标准 Transformer 编码器堆叠
        self.layers = nn.ModuleList([
            Transformer_EncoderLayer_Standard(d_model, nhead) 
            for _ in range(num_layers)
        ])
        
        # 保持一致的多任务输出头
        self.head_horizon = nn.Sequential(
            nn.Linear(d_model, 1024), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(1024, horizon * input_dim), nn.Sigmoid()
        )
        self.head_type = nn.Sequential(
            nn.Linear(d_model * 2, 512), nn.LayerNorm(512), nn.GELU(),
            nn.Dropout(0.4), nn.Linear(512, 5)
        )
        self.head_physics = nn.Sequential(nn.Linear(d_model, 256), nn.GELU(), nn.Linear(256, 2))

    def forward(self, x):
        # MSFF 预处理
        x_in = x.permute(0, 2, 1)
        x_feat = torch.cat([self.conv3(x_in), self.conv5(x_in), self.conv7(x_in)], dim=1)
        x = self.gelu(self.bn(x_feat)).permute(0, 2, 1)
        
        # Transformer 编码过程
        x = x + self.pos_emb
        for layer in self.layers:
            x = layer(x)
            
        # 多尺度特征聚合 (Mean + Max)
        avg_feat = torch.mean(x, dim=1)
        max_feat, _ = torch.max(x, dim=1)
        concat_feat = torch.cat([avg_feat, max_feat], dim=1)
        feat_last = x[:, -1, :] 
        
        return self.head_horizon(feat_last).view(-1, self.horizon, 10), \
               self.head_type(concat_feat), \
               self.head_physics(avg_feat)

# ---------------------- 4. 训练程序 (对齐 Proposed 参数) ----------------------

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.gamma = gamma; self.ce = nn.CrossEntropyLoss(label_smoothing=0.15)
    def forward(self, input, target):
        logp = self.ce(input, target); p = torch.exp(-logp)
        return ((1 - p) ** self.gamma * logp).mean()

def run_transformer_baseline_train(data_path, save_h5_path, save_pth_path):
    with h5py.File(data_path, 'r') as f:
        X = torch.FloatTensor(f['X'][:])
        Y_h = torch.FloatTensor(f['Y_horizon'][:])
        Y_t = torch.LongTensor(f['gt_type'][:])
        P_true = torch.FloatTensor(np.stack([np.mean(Y_h.numpy(), axis=(1,2)), np.sum(Y_h.numpy(), axis=(1,2)) * 0.05], axis=1))

    loader = DataLoader(TensorDataset(X, Y_h, Y_t, P_true), batch_size=32, shuffle=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 初始化基准模型
    model = Transformer_Satellite_Baseline(seq_len=12).to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=3e-2)
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=4e-4, 
                                              steps_per_epoch=len(loader), epochs=60,
                                              pct_start=0.2, div_factor=10)
    
    criterion_bce, criterion_focal, criterion_mse = nn.BCELoss(), FocalLoss(), nn.MSELoss()

    print(f"🚀 [Baseline] 启动标准 Transformer 模型训练...")
    losses = []
    for epoch in range(60):
        model.train(); total_l = 0
        for bx, byh, byt, bp in loader:
            bx, byh, byt, bp = bx.to(device), byh.to(device), byt.to(device), bp.to(device)
            optimizer.zero_grad()
            ph, pt, pp = model(bx)
            loss = 1.0 * criterion_bce(ph, byh) + 5.0 * criterion_focal(pt, byt) + 0.5 * criterion_mse(pp, bp)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.8)
            optimizer.step()
            scheduler.step()
            total_l += loss.item()
        
        losses.append(total_l/len(loader))
        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}/60 | 均值 Loss: {losses[-1]:.4f}")

    # 保存权重用于后续对比
    os.makedirs(os.path.dirname(save_pth_path), exist_ok=True)
    torch.save(model.state_dict(), save_pth_path)
    
    # 最终评估
    model.eval(); all_h, all_t = [] ,[]
    with torch.no_grad():
        for bx, _, _, _ in DataLoader(TensorDataset(X, Y_h, Y_t, P_true), batch_size=32):
            ph, pt, _ = model(bx.to(device))
            all_h.append(ph.cpu().numpy()); all_t.append(pt.cpu().numpy())
    
    fh, ft = np.concatenate(all_h), np.concatenate(all_t)
    f1 = f1_score(Y_h.numpy().flatten() > 0.5, fh.flatten() > 0.4)
    acc = np.mean(Y_t.numpy() == np.argmax(ft, axis=1))

    print("\n" + "📊" * 15 + "\n【Transformer 基准模型评估结果】")
    print(f"🔹 预测态势 F1: {f1:.4f} (对照 Proposed FEDformer: 0.9803)\n🔹 种类识别 Acc: {acc*100:.2f}% (对照 Proposed FEDformer: 93.69%)")
    print("📊" * 15)

    with h5py.File(save_h5_path, 'w') as f:
        f.create_dataset("predicted_situations", data=fh)

if __name__ == "__main__":
    IN = "/root/autodl-tmp/validate/0218/Prediction/sim_dataset_v6_5/academic_long_horizon_v6_5_200k.h5"
    OUT_H5 = "/root/autodl-tmp/validate/0218/Prediction/Transformer/baseline_results.h5"
    OUT_PTH = "/root/autodl-tmp/validate/0218/Prediction/Transformer/best_baseline_transformer.pth"
    run_transformer_baseline_train(IN, OUT_H5, OUT_PTH)

🚀 [Baseline] 启动标准 Transformer 模型训练...
Epoch 10/60 | 均值 Loss: 3.0272
Epoch 20/60 | 均值 Loss: 1.9971
Epoch 30/60 | 均值 Loss: 1.2975
Epoch 40/60 | 均值 Loss: 0.9633
Epoch 50/60 | 均值 Loss: 0.7965
Epoch 60/60 | 均值 Loss: 0.7579

📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊
【Transformer 基准模型评估结果】
🔹 预测态势 F1: 0.9715 (对照 Proposed FEDformer: 0.9803)
🔹 种类识别 Acc: 93.67% (对照 Proposed FEDformer: 93.69%)
📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊
